In [ ]:
from Strategies.Models.Model_trades import TradesModel, TradesModelRaw, TradesModelCandles
from Math.ti_class import TI_class, TR_class
from Strategies.Backtester_class import BacktesterClass
from Strategies.Strategy_class import StrategyVolatility1, StrategyMeanReversion
import plotly_express as px
from Strategies.Calibration_class import Calibration
import numpy as np
from Database.TPData import TPData
from Utilities.plotUtils import bokehPlot
import pickle
import pandas as pd
from datetime import  time


In [ ]:
def to_numeric(string):
    try:
        if '.' in string:
            return pd.to_numeric(string.replace(',','').replace(' ',''))
        else:
            return pd.to_numeric(string.replace(',','.').replace(' ',''))
    except:
        return np.nan
    
df = pd.read_csv(r'S:\Algo\Database\Backups\EUA_sample3.csv', skiprows=2, delimiter= ';')
df.columns = ['datetime', 'price', 'volume', 'type']
data_raw = df.loc[df['type'] == 'TRADE', ['datetime', 'price', 'volume']].copy()
data_raw = data_raw.sort_values(by='datetime')
data_raw['datetime'] = pd.to_datetime(data_raw['datetime'])
data_raw['price'] = data_raw['price'].map(to_numeric)
data_raw['volume'] = data_raw['volume'].map(pd.to_numeric)

data_raw = data_raw[data_raw['volume'] < 20]


In [ ]:
with open(r'X:\Strategies\test\de_market_2023.pkl', 'rb') as pickle_file:
    data_raw = pickle.load(pickle_file)['dem1']

data_class = TPData()
data_p = data_class.filter_data(data_raw, data_raw['price'], 20)
data_p.dropna(subset=['price'], inplace=True)
data_p.reset_index(inplace=True)

In [ ]:

def ret_series(data_series, idx_series):
    df_ret = pd.DataFrame([])
    agg_dict = agg_dict = {'index': 'first', data_series.name: 'mean'}
    data_aux = pd.concat([data_series.reindex(idx_series.index), idx_series],
                         axis=1).reset_index()
    data_aux = data_aux.groupby(0).agg(agg_dict).set_index('index')
    grouped = data_aux.groupby(data_aux.index.date)
    data_dict = {date: group for date, group in grouped}
    for data in data_dict.values():
        ret_aux = np.log(data.fillna(method='ffill')).diff()
        df_ret = pd.concat([df_ret, ret_aux])
    return df_ret.dropna()

agg_dict = {'price': 'mean', 'volume': 'sum'}
data_p = data_raw.set_index('datetime')
start_time = time(8, 0, 0)
end_time = time(18, 0, 0)
data_p = data_p.between_time(start_time, end_time)

# Expected size of candle
data_class = TPData()
data_p = data_class.filter_data(data_p, data_p['price'], 20)
data_p = data_p.groupby(data_p.index).agg(agg_dict)

tau = 85
# EMA
tau_ema = 50
ti_cls = TR_class(tau, tau_ema)
candles = ti_cls.plot_candles(data_p.iloc[:, 0], data_p.iloc[:, 0], volume_series=data_p.iloc[:,1])
candles['p_avg'] = (candles['Open'] + candles['Close'])/2


In [ ]:
from scipy.stats import norm
import matplotlib.pyplot as plt

idx_series = ti_cls.tick_imbalance_indices(data_p.iloc[:, 0])
grouped = idx_series.groupby(idx_series).count()
#grouped.plot(kind='hist', bins=range(1, grouped.max(), int(grouped.max() / 50)))
print('mean: %s, median: %s ' % (grouped.mean(), grouped.median()))

# Calculate distribution of returns
ret = ret_series(data_p.iloc[:, 0], idx_series)
#ret.plot(kind='hist', bins=np.arange(ret.min()[0], ret.max()[0], ret.max()[0] / 50))
plt.figure()
plt.hist(ret, bins=50, density=True, alpha=0.6, color='g')
# Calculate mean and standard deviation
mu, std = ret.mean()[0], ret.std()[0]
skw, kur = ret.skew()[0], ret.kurtosis()[0]
x = np.linspace(ret.min()[0] - std, ret.max()[0] + std, 100)
p = norm.pdf(x, mu, std)
plt.plot(x, p, 'k', linewidth=2)
plt.show()

print('mean: %s, std: %s, skew: %s, curtosis: %s ' % (mu, std, skw, kur))

grouped = idx_series.groupby(idx_series.index.date).agg(['first', 'last'])
grouped = grouped.iloc[:, 1] - grouped.iloc[:, 0] + 1
print('mean: %s, median: %s ' % (grouped.mean(), grouped.median()))

In [ ]:
import matplotlib.pyplot as plt
plt.figure()
plt.hist(df_data['count'], bins=10, density=True, alpha=0.6, color='g')

In [ ]:
def run_backTest_w_params(df_input, param_val):
    back_test = BacktesterClass()
    model = TradesModelCandles(params=param_val, comp_col='Close')
    strat = StrategyMeanReversion(trade_col='p_avg', comp_col='Close',params=param_val)
    output_series = back_test.simulate_strategy(df_input, model, strat)
    return strat.data_df, output_series, pd.DataFrame(back_test.pnl_dict), strat.stats_dict

candles.reset_index(inplace=True)

In [ ]:
df_data, returns, trades_data, strat = run_backTest_w_params(candles, np.array([20, 1.5, 1, .7, 1, 10]))
trades_data.set_index('index',inplace=True)
df_data.set_index('index',inplace=True)
df_data.reset_index(drop=True,inplace=True)
df= pd.concat([df_data, trades_data[['pnl', 'trend', 'take_profit', 'stop_loss']]], axis=1)
df['returns'] = df['pnl'].fillna(0).cumsum()
bokehPlot(df, title="strategy sl/tp fixed by position",
           col_list=[['Close', 'ema', 'take_profit', 'stop_loss', 'openUp', 'openDown', 'emalong'], ['returns']],
           scatter=['take_profit', 'stop_loss'], sub=2)
returns.cumsum().plot()
print('pnl ratio', len(returns[returns > 0])/len(returns))
print('profit mean/ loss mean', returns[returns > 0].mean(), returns[returns < 0].mean())


In [ ]:
pnls = pd.concat([df_data, trades_data[['pnl', 'trend', 'take_profit', 'stop_loss', 'vol']]], axis=1, join='inner')
trades = pnls.copy()
trades['trend'] = trades['trend'].replace(0, method='bfill')
trades = trades[trades['pnl'] != 0]

In [ ]:
trades['abs_trend'] = abs(trades['trend'] - 1)

In [ ]:
import seaborn as sns
sns.regplot(trades[(trades['trend'] > 0) & (trades['sigma'] < .9)], x='abs_trend', y='pnl', order=1)

In [ ]:
import plotly.express as px
import plotly.graph_objects as go

# Create the 3D scatter plot
fig = px.scatter_3d(trades[trades['sigma'] < 1], x='trend', y='pnl', z='sigma')

# Add labels and a title
fig.update_layout(scene=dict(xaxis_title='X trend', yaxis_title='Y pnl', zaxis_title='Z sigma'),
                  scene_aspectmode='cube')

# Create a regression plane
X = trades[trades['sigma'] < 1]['trend']
Y = trades[trades['sigma'] < 1]['pnl']
Z = trades[trades['sigma'] < 1]['sigma']

A = np.vstack([X, Z, np.ones(len(X))]).T
model, _, _, _ = np.linalg.lstsq(A, Y, rcond=None)
xx, zz = np.meshgrid(X, Z)
yy = model[0] * xx + model[1] * zz + model[2]

fig.add_trace(go.Surface(x=xx, y=yy, z=zz, opacity=0.8))

# Show the plot
fig.show()

In [ ]:
filtered = trades[(trades['sigma'] < .45) & (trades['trend'] <= 1.02) & (trades['trend'] >= .98)]
len(filtered)


In [ ]:
filtered['pnl'].cumsum().plot()
print(np.mean(filtered['pnl'])/np.std(filtered['pnl']))
print(np.mean(trades['pnl'])/np.std(trades['pnl']))

In [ ]:
diff